In [20]:
# ============================================================
# Cell 1: Install Libraries
# ============================================================
!pip install -q openai-whisper
!pip install -q jiwer
!pip install -q gradio
!pip install -q librosa soundfile
!pip install -q kaggle
!apt-get install -y -q ffmpeg
print("✅ All libraries installed.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
✅ All libraries installed.


In [21]:
# ============================================================
# Cell 2: Kaggle Dataset Path
# ============================================================
import os

DATA_DIR = "/kaggle/input/datasets/mayarjao/arabic-tts"
print(f"✅ Using dataset path: {DATA_DIR}")
print("Contents:")
for item in os.listdir(DATA_DIR):
    print(f"  {item}")


✅ Using dataset path: /kaggle/input/datasets/mayarjao/arabic-tts
Contents:
  arabic_tts


In [22]:
# ============================================================
# Cell 3: Explore Dataset Structure
# ============================================================
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/mayarjao/arabic-tts"

# Walk the folder structure to understand what we have
for root, dirs, files_list in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Show files only 2 levels deep
        sub_indent = '  ' * (level + 1)
        for f in files_list[:10]:  # First 10 files only
            print(f"{sub_indent}{f}")
        if len(files_list) > 10:
            print(f"{sub_indent}... and {len(files_list)-10} more files")

arabic-tts/
  arabic_tts/
    metadata-wav.csv
    metadata.csv
    wavs/


In [23]:
# ============================================================
# Cell 3b: Load the metadata CSV
# ============================================================
import pandas as pd
import os

DATA_DIR = "/kaggle/input/datasets/mayarjao/arabic-tts"

# Find whichever CSV files exist
csv_files = []
for root, dirs, files_list in os.walk(DATA_DIR):
    for f in files_list:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

print("Found CSV files:")
for t in csv_files:
    print(f"  {t}")

# Load the main metadata file (prioritize 'metadata.csv')
METADATA_PATH = None
for f_path in csv_files:
    if "metadata.csv" in os.path.basename(f_path):
        METADATA_PATH = f_path
        break

if not METADATA_PATH and csv_files:
    METADATA_PATH = csv_files[0] # Fallback to the first found CSV if metadata.csv not explicit

if METADATA_PATH:
    print(f"\nLoading metadata from: {METADATA_PATH}")
    # The file appears to be pipe-separated and lacks a header row.
    # Using header=None and then assigning column names.
    df = pd.read_csv(METADATA_PATH, sep='|', header=None)
    df.columns = ['text', 'audio_file']

    print(f"\n✅ Loaded metadata: {len(df)} rows")
    print(f"Columns: {list(df.columns)}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
else:
    print("❌ No metadata CSV file found.")

Found CSV files:
  /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/metadata-wav.csv
  /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/metadata.csv

Loading metadata from: /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/metadata.csv

✅ Loaded metadata: 78720 rows
Columns: ['text', 'audio_file']

First 3 rows:
                           text                    audio_file
0    هو يحبّها و هي تحبّه أيضا.  common_voice_ar_24203362.wav
1  من الممكن أنها لن تأتي غداً.  common_voice_ar_22931432.wav
2                     إبنك بطل.  common_voice_ar_26338992.wav


In [24]:
# ============================================================
# Cell 4: Build File Index + Train/Val/Test Split
# ============================================================
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Find the audio clips folder
# Common Voice structure: clips/ folder with .mp3 files
CLIPS_DIR = None
for root, dirs, files_list in os.walk(DATA_DIR):
    # The audio files are .wav in the 'wavs' subdirectory
    if any(f.endswith('.wav') for f in files_list):
        CLIPS_DIR = root
        break

print(f"Audio clips folder: {CLIPS_DIR}")

# Build clean dataframe: only rows where audio file actually exists
def build_clean_df(df, clips_dir):
    """Keep only rows where the audio file exists on disk."""
    records = []
    for _, row in df.iterrows():
        # Use 'audio_file' column for the filename and 'text' for the sentence
        fname = row['audio_file']
        sentence = str(row['text']).strip()

        # Construct the full path to the audio file
        audio_path = os.path.join(clips_dir, fname)

        if os.path.exists(audio_path):
            records.append({
                'audio_path': audio_path,
                'sentence': sentence
            })
    return pd.DataFrame(records)

clean_df = build_clean_df(df, CLIPS_DIR)
print(f"✅ Valid audio+text pairs: {len(clean_df)}")

# Drop rows with empty transcripts
clean_df = clean_df[clean_df['sentence'].str.len() > 2].reset_index(drop=True)
print(f"✅ After removing empty transcripts: {len(clean_df)}")

# Train / Validation / Test split: 70 / 15 / 15
train_df, temp_df = train_test_split(clean_df, test_size=0.30, random_state=42)
val_df, test_df   = train_test_split(temp_df,  test_size=0.50, random_state=42)

print(f"\nSplit sizes:")
print(f"  Train : {len(train_df)}")
print(f"  Val   : {len(val_df)}")
print(f"  Test  : {len(test_df)}")

# Save splits to CSV for reproducibility
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv",   index=False)
test_df.to_csv("/content/test.csv",  index=False)
print("\n✅ Splits saved to /content/train.csv, val.csv, test.csv")

Audio clips folder: /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/wavs
✅ Valid audio+text pairs: 78720
✅ After removing empty transcripts: 78687

Split sizes:
  Train : 55080
  Val   : 11803
  Test  : 11804

✅ Splits saved to /content/train.csv, val.csv, test.csv


In [25]:
# ============================================================
# Cell 5: Preprocessing — Load & Normalize Audio
# ============================================================
import librosa
import numpy as np
import re

TARGET_SR = 16000  # Whisper and Wav2Vec both require 16kHz

def load_audio(path, target_sr=TARGET_SR):
    """
    Load any .mp3 or .wav file as a 16kHz mono float32 numpy array.
    librosa handles format conversion automatically via ffmpeg.
    """
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio.astype(np.float32)

def normalize_audio(audio):
    """Scale amplitude so peak = 1.0"""
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
    return audio

def normalize_arabic_text(text):
    """
    Clean Arabic text before WER calculation.
    - Remove diacritics (tashkeel / harakat)
    - Normalize alef variants (أ إ آ → ا)
    - Remove punctuation
    - Collapse whitespace
    """
    # Remove Arabic diacritics
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    # Normalize alef
    text = re.sub(r'[أإآ]', 'ا', text)
    # Remove tatweel
    text = re.sub(r'\u0640', '', text)
    # Keep only Arabic letters and spaces
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Normalize whitespace
    return ' '.join(text.split()).strip()

# Quick test on one sample
sample = test_df.iloc[0]
audio  = load_audio(sample['audio_path'])
audio  = normalize_audio(audio)
ref    = normalize_arabic_text(sample['sentence'])

print(f"Audio duration : {len(audio)/TARGET_SR:.2f} seconds")
print(f"Audio shape    : {audio.shape}")
print(f"Reference text : {ref}")

Audio duration : 2.92 seconds
Audio shape    : (46657,)
Reference text : وربما كانت ضارة


In [ ]:
# ============================================================
# Cell ED-1: Emotion Detection — Install Extra Dependencies
# ============================================================
!pip install -q scikit-learn imbalanced-learn joblib
print("✅ Emotion detection dependencies installed.")


In [ ]:
# ============================================================
# Cell ED-2: Emotion Detection — Audio Feature Extraction
# ============================================================
import numpy as np
import librosa

TARGET_SR = 16000  # already defined above; redefined here for safety

# Expected feature vector length — used to pad on error
_N_FEATURES = 187


def extract_emotion_features(audio: np.ndarray, sr: int = TARGET_SR) -> np.ndarray:
    """
    Extract a fixed-length 187-dim feature vector from a raw audio array.
    Every sub-feature is individually guarded so a single failure
    (silent clip, too-short clip, NaN from PYIN) never crashes the pipeline.

    Features
    --------
    MFCCs (40)       mean + std  →  80 dims
    Delta-MFCCs (40) mean + std  →  80 dims
    Chroma (12)      mean        →  12 dims
    Spectral contrast (n_bands=6 → 7 rows) mean → 7 dims
    ZCR              mean + std  →   2 dims
    RMS              mean + std  →   2 dims
    F0 (PYIN)        mean + std  →   2 dims
    Spectral rolloff mean        →   1 dim
    Spectral centroid mean       →   1 dim
    Total                        → 187 dims
    """
    # Guard: empty or near-silent clip
    if audio is None or len(audio) < 512:
        return np.zeros(_N_FEATURES, dtype=np.float32)

    feats = []

    # ── MFCCs ──────────────────────────────────────────────────────────────
    try:
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        feats.extend(np.nan_to_num(np.mean(mfcc, axis=1)).tolist())
        feats.extend(np.nan_to_num(np.std(mfcc,  axis=1)).tolist())
    except Exception:
        feats.extend([0.0] * 80)

    # ── Delta-MFCCs ────────────────────────────────────────────────────────
    try:
        mfcc_for_delta = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        delta_mfcc = librosa.feature.delta(mfcc_for_delta)
        feats.extend(np.nan_to_num(np.mean(delta_mfcc, axis=1)).tolist())
        feats.extend(np.nan_to_num(np.std(delta_mfcc,  axis=1)).tolist())
    except Exception:
        feats.extend([0.0] * 80)

    # ── Chroma ─────────────────────────────────────────────────────────────
    try:
        chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_chroma=12)
        feats.extend(np.nan_to_num(np.mean(chroma, axis=1)).tolist())
    except Exception:
        feats.extend([0.0] * 12)

    # ── Spectral contrast (n_bands=6 produces 7 rows) ──────────────────────
    try:
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
        feats.extend(np.nan_to_num(np.mean(contrast, axis=1)).tolist())
    except Exception:
        feats.extend([0.0] * 7)

    # ── Zero-Crossing Rate ─────────────────────────────────────────────────
    try:
        zcr = librosa.feature.zero_crossing_rate(audio)
        feats.append(float(np.nan_to_num(np.mean(zcr))))
        feats.append(float(np.nan_to_num(np.std(zcr))))
    except Exception:
        feats.extend([0.0, 0.0])

    # ── RMS energy ─────────────────────────────────────────────────────────
    try:
        rms = librosa.feature.rms(y=audio)
        feats.append(float(np.nan_to_num(np.mean(rms))))
        feats.append(float(np.nan_to_num(np.std(rms))))
    except Exception:
        feats.extend([0.0, 0.0])

    # ── Fundamental frequency (F0 via PYIN) ────────────────────────────────
    # Guard: PYIN needs at least 2048 samples; filter NaNs from voiced frames
    try:
        if len(audio) >= 2048:
            f0, voiced_flag, _ = librosa.pyin(audio, fmin=50, fmax=500, sr=sr)
            if voiced_flag is not None and f0 is not None:
                voiced_f0 = f0[voiced_flag.astype(bool)]
                voiced_f0 = voiced_f0[np.isfinite(voiced_f0)]  # remove NaNs
            else:
                voiced_f0 = np.array([])
            feats.append(float(np.mean(voiced_f0)) if len(voiced_f0) > 0 else 0.0)
            feats.append(float(np.std(voiced_f0))  if len(voiced_f0) > 0 else 0.0)
        else:
            feats.extend([0.0, 0.0])
    except Exception:
        feats.extend([0.0, 0.0])

    # ── Spectral rolloff ───────────────────────────────────────────────────
    try:
        rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
        feats.append(float(np.nan_to_num(np.mean(rolloff))))
    except Exception:
        feats.append(0.0)

    # ── Spectral centroid ──────────────────────────────────────────────────
    try:
        centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
        feats.append(float(np.nan_to_num(np.mean(centroid))))
    except Exception:
        feats.append(0.0)

    vec = np.array(feats, dtype=np.float32)

    # Pad or trim to exactly _N_FEATURES so shape is always consistent
    if len(vec) < _N_FEATURES:
        vec = np.pad(vec, (0, _N_FEATURES - len(vec)))
    elif len(vec) > _N_FEATURES:
        vec = vec[:_N_FEATURES]

    return np.nan_to_num(vec)   # final NaN/Inf sweep


print("✅ Feature extractor defined.")
print(f"   Feature vector length: {len(extract_emotion_features(np.zeros(TARGET_SR, dtype=np.float32)))}")


In [ ]:
# ============================================================
# Cell ED-3: Emotion Detection — Acoustic Pseudo-Labelling
# ============================================================
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm

EMOTIONS = ["neutral", "happy", "sad", "angry", "fearful"]
MODEL_DIR = "/kaggle/working"   # writable directory on Kaggle


def acoustic_pseudo_label(audio: np.ndarray, sr: int = TARGET_SR) -> str:
    """
    Assign a coarse emotion label from raw acoustic measurements.
    Returns one of: neutral, happy, sad, angry, fearful.
    Fully guarded — never raises.
    """
    try:
        rms_val = float(np.sqrt(np.mean(audio ** 2)))
    except Exception:
        rms_val = 0.0

    try:
        zcr_val = float(np.mean(librosa.feature.zero_crossing_rate(audio)))
    except Exception:
        zcr_val = 0.0

    try:
        if len(audio) >= 2048:
            f0, voiced_flag, _ = librosa.pyin(audio, fmin=50, fmax=500, sr=sr)
            if voiced_flag is not None and f0 is not None:
                voiced_f0 = f0[voiced_flag.astype(bool)]
                voiced_f0 = voiced_f0[np.isfinite(voiced_f0)]
            else:
                voiced_f0 = np.array([])
            f0_mean = float(np.mean(voiced_f0)) if len(voiced_f0) > 3 else 150.0
            f0_std  = float(np.std(voiced_f0))  if len(voiced_f0) > 3 else 20.0
        else:
            f0_mean, f0_std = 150.0, 20.0
    except Exception:
        f0_mean, f0_std = 150.0, 20.0

    high_energy = rms_val  > 0.05
    high_zcr    = zcr_val  > 0.08
    high_f0     = f0_mean  > 180
    high_f0_var = f0_std   > 40

    if high_energy and high_f0 and not high_f0_var:
        return "happy"
    if high_energy and high_zcr and high_f0_var:
        return "angry"
    if not high_energy and high_zcr and high_f0_var:
        return "fearful"
    if not high_energy and not high_zcr:
        return "sad"
    return "neutral"


def build_emotion_dataset(df: pd.DataFrame, max_samples: int = None) -> tuple:
    """
    Extract features + pseudo-labels for every row in df.
    Returns (X, y) numpy arrays ready for sklearn.
    Never raises — returns empty arrays on total failure.
    """
    if max_samples:
        df = df.head(max_samples).copy()

    X, y = [], []
    skipped = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building emotion dataset"):
        try:
            audio = load_audio(row['audio_path'])
            audio = normalize_audio(audio)

            # Skip if audio is empty or too short to be useful
            if audio is None or len(audio) < 512:
                skipped += 1
                continue

            feats = extract_emotion_features(audio)
            label = acoustic_pseudo_label(audio)

            # Skip if features still contain non-finite values
            if not np.isfinite(feats).all():
                skipped += 1
                continue

            X.append(feats)
            y.append(label)
        except Exception:
            skipped += 1

    print(f"\n✅ Dataset built: {len(X)} samples, {skipped} skipped.")

    if len(X) == 0:
        print("⚠️  No valid samples — returning empty arrays.")
        return np.zeros((0, _N_FEATURES), dtype=np.float32), np.array([], dtype=str)

    return np.array(X, dtype=np.float32), np.array(y)


print("Building emotion training data from train_df...")
X_train_emo, y_train_emo = build_emotion_dataset(train_df, max_samples=500)
print("Building emotion validation data from val_df...")
X_val_emo,   y_val_emo   = build_emotion_dataset(val_df,   max_samples=100)
print("Building emotion test data from test_df...")
X_test_emo,  y_test_emo  = build_emotion_dataset(test_df,  max_samples=100)

print("\nLabel distribution (train):")
print(pd.Series(y_train_emo).value_counts().to_string())


In [ ]:
# ============================================================
# Cell ED-4: Emotion Detection — Train MLP Classifier
# ============================================================
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.neural_network  import MLPClassifier
from sklearn.metrics         import classification_report
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling  import SMOTE
from collections import Counter
import numpy as np, joblib, os

# ── Guard: abort gracefully if dataset is empty ─────────────────────────────
if len(X_train_emo) == 0:
    print("❌ Training data is empty — skipping emotion model training.")
else:

    # ── 1. Encode labels ────────────────────────────────────────────────────
    le = LabelEncoder()
    le.fit(EMOTIONS)  # fit on all 5 so label indices are always consistent

    # Safe transform: map any label not in EMOTIONS to "neutral"
    def safe_transform(labels):
        known = set(le.classes_)
        cleaned = [l if l in known else "neutral" for l in labels]
        return le.transform(cleaned)

    y_tr = safe_transform(y_train_emo)
    y_va = safe_transform(y_val_emo)   if len(y_val_emo)  > 0 else np.array([], dtype=int)
    y_te = safe_transform(y_test_emo)  if len(y_test_emo) > 0 else np.array([], dtype=int)

    # ── 2. Feature scaling ──────────────────────────────────────────────────
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_emo)
    X_va_s = scaler.transform(X_val_emo)  if len(X_val_emo)  > 0 else X_val_emo
    X_te_s = scaler.transform(X_test_emo) if len(X_test_emo) > 0 else X_test_emo

    # ── 3. SMOTE — dynamic k_neighbors to survive small classes ─────────────
    class_counts = Counter(y_tr)
    n_classes    = len(class_counts)
    min_class_n  = min(class_counts.values())

    print(f"Classes in train: {dict(class_counts)}")

    if n_classes < 2:
        print("⚠️  Only 1 class found — skipping SMOTE, training on raw data.")
        X_tr_bal, y_tr_bal = X_tr_s, y_tr
    else:
        k_neighbors = max(1, min(3, min_class_n - 1))
        print(f"Applying SMOTE (k_neighbors={k_neighbors})...")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        try:
            X_tr_bal, y_tr_bal = smote.fit_resample(X_tr_s, y_tr)
            print(f"   Before SMOTE: {len(X_tr_s)} | After SMOTE: {len(X_tr_bal)}")
        except Exception as e:
            print(f"⚠️  SMOTE failed ({e}) — using original data.")
            X_tr_bal, y_tr_bal = X_tr_s, y_tr

    # ── 4. Train MLP ─────────────────────────────────────────────────────────
    print("\nTraining MLP classifier...")
    clf = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=0.005,
        alpha=0.01,
        batch_size=min(128, len(X_tr_bal)),  # batch_size <= n_samples
        learning_rate="adaptive",
        max_iter=50,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=8,
        random_state=42,
        verbose=False,
    )
    clf.fit(X_tr_bal, y_tr_bal)
    print(f"   Stopped after {clf.n_iter_} iterations.")

    # ── 5. Cross-validation — dynamic folds ─────────────────────────────────
    min_class_after = min(Counter(y_tr_bal).values())
    cv_folds = max(2, min(3, min_class_after))
    print(f"\nRunning {cv_folds}-fold cross-validation...")
    try:
        cv_scores = cross_val_score(clf, X_tr_bal, y_tr_bal,
                                    cv=cv_folds, scoring="accuracy", n_jobs=-1)
        print(f"   CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
    except Exception as e:
        print(f"⚠️  Cross-validation failed: {e}")

    # ── 6. Evaluate on val and test ──────────────────────────────────────────
    for split_name, X_s, y_s in [("Validation", X_va_s, y_va), ("Test", X_te_s, y_te)]:
        if len(y_s) == 0:
            print(f"\n{split_name}: no samples — skipping.")
            continue
        preds = clf.predict(X_s)
        acc   = np.mean(preds == y_s) * 100
        present_labels = sorted(set(y_s.tolist()) | set(preds.tolist()))
        present_names  = le.inverse_transform(present_labels)
        print(f"\n{'='*50}")
        print(f"  {split_name} Accuracy: {acc:.2f}%")
        print(classification_report(y_s, preds,
                                     labels=present_labels,
                                     target_names=present_names,
                                     zero_division=0))

    # ── 7. Save artefacts ────────────────────────────────────────────────────
    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(clf,    os.path.join(MODEL_DIR, "emotion_clf.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_DIR, "emotion_scaler.joblib"))
    joblib.dump(le,     os.path.join(MODEL_DIR, "emotion_le.joblib"))
    print(f"\n✅ Emotion model saved to {MODEL_DIR}/")


In [ ]:
# ============================================================
# Cell ED-5: Emotion Detection — Inference Helper
# ============================================================
import numpy as np
import joblib
import os

# Emoji map — defined here AND kept available for Gradio cell
_EMOTION_EMOJI = {
    "happy":   "😊",
    "sad":     "😢",
    "angry":   "😠",
    "fearful": "😨",
    "neutral": "😐",
}

# Load saved artefacts safely
_emo_clf = _emo_scaler = _emo_le = None
try:
    _emo_clf    = joblib.load(os.path.join(MODEL_DIR, "emotion_clf.joblib"))
    _emo_scaler = joblib.load(os.path.join(MODEL_DIR, "emotion_scaler.joblib"))
    _emo_le     = joblib.load(os.path.join(MODEL_DIR, "emotion_le.joblib"))
    print("✅ Emotion model artefacts loaded.")
except Exception as e:
    print(f"⚠️  Could not load emotion model: {e}")
    print("   Run Cell ED-4 first to train and save the model.")


def predict_emotion(audio: np.ndarray, sr: int = TARGET_SR) -> tuple:
    """
    Predict emotion from a raw 16kHz mono float32 audio array.

    Returns
    -------
    label : str   — predicted emotion label
    proba : dict  — {emotion: probability} for all 5 classes
    """
    if _emo_clf is None or _emo_scaler is None or _emo_le is None:
        return "neutral", {e: 0.0 for e in EMOTIONS}

    try:
        feats   = extract_emotion_features(audio, sr)
        feats_s = _emo_scaler.transform(feats.reshape(1, -1))
        pred_idx  = _emo_clf.predict(feats_s)[0]
        label     = _emo_le.inverse_transform([pred_idx])[0]
        proba_arr = _emo_clf.predict_proba(feats_s)[0]
        # Build full 5-class proba dict (pad missing classes with 0.0)
        proba = {e: 0.0 for e in EMOTIONS}
        for i, p in enumerate(_emo_clf.classes_):
            proba[_emo_le.inverse_transform([p])[0]] = float(proba_arr[i])
        return label, proba
    except Exception as e:
        print(f"⚠️  predict_emotion failed: {e}")
        return "neutral", {e: 0.0 for e in EMOTIONS}


# ── Quick sanity check ───────────────────────────────────────────────────────
if _emo_clf is not None and len(test_df) > 0:
    print("Quick emotion inference demo:")
    print("-" * 50)
    for _, row in test_df.head(min(3, len(test_df))).iterrows():
        try:
            audio = normalize_audio(load_audio(row["audio_path"]))
            label, proba = predict_emotion(audio)
            emoji  = _EMOTION_EMOJI.get(label, "")
            top3   = sorted(proba.items(), key=lambda x: -x[1])[:3]
            top3_str = ", ".join(f"{e}: {p*100:.1f}%" for e, p in top3)
            print(f"  Predicted : {emoji} {label}")
            print(f"  Top-3     : {top3_str}")
            print(f"  Text ref  : {row['sentence'][:60]}")
            print()
        except Exception as e:
            print(f"  ⚠️  Sample failed: {e}")
else:
    print("⚠️  Skipping demo — model not loaded or test_df empty.")


In [26]:
# ============================================================
# Cell 6: Load Whisper Model
# ============================================================
import whisper
import torch

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

# Model size guide:
#   tiny   → fastest, lowest accuracy (~WER 35–50% on Arabic)
#   small  → good balance for testing
#   medium → recommended for final submission (fits on free Colab T4)
#   large-v3 → best accuracy, needs Colab Pro or batch_size=1

MODEL_SIZE = "medium"  # Change to "large-v3" if you have Colab Pro
model = whisper.load_model(MODEL_SIZE, device=device)
print(f"✅ Whisper-{MODEL_SIZE} loaded on {device}")

Device: cuda
✅ Whisper-medium loaded on cuda


In [27]:
# ============================================================
# Cell 7: Batch Inference — Stream Through Dataset
# ============================================================
# We process N clips at a time so we never load the whole dataset into RAM.

import pandas as pd
from tqdm import tqdm

def transcribe_batch(model, df, batch_size=8, max_samples=None):
    """
    Stream through df in batches of batch_size.
    Returns a DataFrame with columns: audio_path, reference, hypothesis
    """
    if max_samples:
        df = df.head(max_samples)

    results = []
    errors  = 0

    for i in tqdm(range(0, len(df), batch_size), desc="Transcribing"):
        batch = df.iloc[i : i + batch_size]

        for _, row in batch.iterrows():
            try:
                audio = load_audio(row['audio_path'])
                audio = normalize_audio(audio)

                result = model.transcribe(
                    audio,
                    language="ar",       # Force Arabic — don't let Whisper guess
                    task="transcribe",   # Keep Arabic (not translate to English)
                    fp16=(device == "cuda"),  # Faster on GPU
                    temperature=0.0,     # Greedy decoding (deterministic)
                )
                hyp = result["text"].strip()

            except Exception as e:
                hyp = ""
                errors += 1

            results.append({
                'audio_path': row['audio_path'],
                'reference' : row['sentence'],
                'hypothesis': hyp
            })

    print(f"\n✅ Done. {len(results)} samples processed, {errors} errors.")
    return pd.DataFrame(results)

# Run on the test set (use max_samples=50 for quick eval, None for full)
print("Running inference on test set...")
results_df = transcribe_batch(model, test_df, batch_size=8, max_samples=100)

# Save results
results_df.to_csv("/content/predictions.csv", index=False)
print("✅ Predictions saved to /content/predictions.csv")

# Preview
print("\nSample predictions:")
for _, row in results_df.head(5).iterrows():
    print(f"  REF: {row['reference']}")
    print(f"  HYP: {row['hypothesis']}")
    print()

Running inference on test set...


Transcribing: 100%|██████████| 13/13 [01:15<00:00,  5.82s/it]


✅ Done. 100 samples processed, 0 errors.
✅ Predictions saved to /content/predictions.csv

Sample predictions:
  REF: وَرُبَّمَا كَانَتْ ضَارَّةً
  HYP: وربما كانت ضارة

  REF: هوايتي زيارة المعابد القديمة.
  HYP: هواية زيارة المعابد القديمة

  REF: وَما يَأتيهِم مِن ذِكرٍ مِنَ الرَّحمنِ مُحدَثٍ إِلّا كانوا عَنهُ مُعرِضينَ
  HYP: وما يأتيهم من ذكر من الرحمن محدث إلا كانوا عنه معرضين

  REF: السمك رخيص اليوم.
  HYP: السمك رخيص اليوم

  REF: لم تعد إليّ نقودي.
  HYP: لم تعود إليه نقودي



In [28]:
# ============================================================
# Cell 8: Evaluation — WER and CER
# ============================================================
from jiwer import wer, cer
import pandas as pd

results_df = pd.read_csv("/content/predictions.csv")

# Normalize both reference and hypothesis before scoring
refs = [normalize_arabic_text(r) for r in results_df['reference'].tolist()]
hyps = [normalize_arabic_text(h) for h in results_df['hypothesis'].tolist()]

# Remove pairs where reference or hypothesis is empty
pairs = [(r, h) for r, h in zip(refs, hyps) if r and h]
refs_clean = [p[0] for p in pairs]
hyps_clean = [p[1] for p in pairs]

word_error_rate = wer(refs_clean, hyps_clean)
char_error_rate = cer(refs_clean, hyps_clean)

print("=" * 45)
print(f"  Model        : Whisper-{MODEL_SIZE}")
print(f"  Test samples : {len(pairs)}")
print(f"  WER          : {word_error_rate*100:.2f}%")
print(f"  CER          : {char_error_rate*100:.2f}%")
print("=" * 45)

# Show best and worst predictions
results_df['ref_norm'] = refs
results_df['hyp_norm'] = hyps
results_df['sample_wer'] = results_df.apply(
    lambda row: wer(row['ref_norm'], row['hyp_norm'])
    if row['ref_norm'] and row['hyp_norm'] else 1.0, axis=1
)

print("\n--- 3 Best Predictions (lowest WER) ---")
for _, row in results_df.nsmallest(3, 'sample_wer').iterrows():
    print(f"  REF: {row['ref_norm']}")
    print(f"  HYP: {row['hyp_norm']}")
    print(f"  WER: {row['sample_wer']:.2f}\n")

print("--- 3 Worst Predictions (highest WER) ---")
for _, row in results_df.nlargest(3, 'sample_wer').iterrows():
    print(f"  REF: {row['ref_norm']}")
    print(f"  HYP: {row['hyp_norm']}")
    print(f"  WER: {row['sample_wer']:.2f}\n")

  Model        : Whisper-medium
  Test samples : 100
  WER          : 26.85%
  CER          : 9.25%

--- 3 Best Predictions (lowest WER) ---
  REF: وربما كانت ضارة
  HYP: وربما كانت ضارة
  WER: 0.00

  REF: وما ياتيهم من ذكر من الرحمن محدث الا كانوا عنه معرضين
  HYP: وما ياتيهم من ذكر من الرحمن محدث الا كانوا عنه معرضين
  WER: 0.00

  REF: السمك رخيص اليوم
  HYP: السمك رخيص اليوم
  WER: 0.00

--- 3 Worst Predictions (highest WER) ---
  REF: حيادي
  HYP: كي يدين
  WER: 2.00

  REF: احدهما ضبط الفرج عن الحرام
  HYP: اهدهم ما ضبطوا الفرتيان الحراني
  WER: 1.00

  REF: وتفارق فراق العجول
  HYP: وتفارغ في راق العجول
  WER: 1.00



In [29]:
# ============================================================
# Cell 9: Experiment — Compare Whisper Model Sizes
# ============================================================
# Compare whisper-small vs whisper-medium on the same 30 clips
# This gives you a real experiment table for your report.

import time

EXPERIMENT_SAMPLES = 30
experiment_df = test_df.head(EXPERIMENT_SAMPLES)
comparison_results = {}

for size in ["small", "medium"]:
    print(f"\nLoading whisper-{size}...")
    exp_model = whisper.load_model(size, device=device)

    refs_exp, hyps_exp = [], []
    t0 = time.time()

    for _, row in experiment_df.iterrows():
        try:
            audio = normalize_audio(load_audio(row['audio_path']))
            result = exp_model.transcribe(audio, language="ar", task="transcribe", temperature=0.0)
            hyp = result["text"].strip()
        except:
            hyp = ""
        refs_exp.append(row['sentence'])
        hyps_exp.append(hyp)

    elapsed = time.time() - t0
    refs_norm_exp = [normalize_arabic_text(r) for r in refs_exp]
    hyps_norm_exp = [normalize_arabic_text(h) for h in hyps_exp]
    pairs_exp = [(r,h) for r,h in zip(refs_norm_exp, hyps_norm_exp) if r and h]

    word_err = wer([p[0] for p in pairs_exp], [p[1] for p in pairs_exp])
    char_err = cer([p[0] for p in pairs_exp], [p[1] for p in pairs_exp])

    comparison_results[size] = {
        'WER (%)': round(word_err * 100, 2),
        'CER (%)': round(char_err * 100, 2),
        'Time (s)': round(elapsed, 1)
    }
    del exp_model  # Free GPU memory
    torch.cuda.empty_cache()

# Print comparison table
print("\n\n" + "=" * 50)
print("  Model Comparison Table")
print("=" * 50)
import pandas as pd
comp_df = pd.DataFrame(comparison_results).T
comp_df.index.name = "Model"
print(comp_df.to_string())
print("=" * 50)


Loading whisper-small...

Loading whisper-medium...


  Model Comparison Table
        WER (%)  CER (%)  Time (s)
Model                             
small     41.08    16.21      11.2
medium    29.73    10.73      22.3


In [30]:
# ============================================================
# Cell 10: Experiment — Clean vs Noisy Audio (Robustness Test)
# ============================================================

def add_gaussian_noise(audio, snr_db=10):
    """
    Add white Gaussian noise at a given SNR (signal-to-noise ratio in dB).
    Lower SNR = more noise. 10dB is noticeably noisy, 20dB is mild.
    """
    signal_power = np.mean(audio ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise        = np.random.normal(0, np.sqrt(noise_power), len(audio))
    return (audio + noise).astype(np.float32)

# Use the already-loaded medium model
N_NOISE_TEST = 20
noise_df = test_df.head(N_NOISE_TEST)

refs_noise, hyps_clean, hyps_noisy = [], [], []

for _, row in noise_df.iterrows():
    try:
        audio = normalize_audio(load_audio(row['audio_path']))
        noisy = add_gaussian_noise(audio, snr_db=10)  # 10dB noise

        res_clean = model.transcribe(audio, language="ar", temperature=0.0)
        res_noisy = model.transcribe(noisy, language="ar", temperature=0.0)

        refs_noise.append(normalize_arabic_text(row['sentence']))
        hyps_clean.append(normalize_arabic_text(res_clean['text']))
        hyps_noisy.append(normalize_arabic_text(res_noisy['text']))
    except:
        pass

wer_clean = wer(refs_noise, hyps_clean) * 100
wer_noisy = wer(refs_noise, hyps_noisy) * 100

print(f"\nRobustness Experiment (N={len(refs_noise)}, SNR=10dB noise)")
print(f"{'Condition':<20} {'WER':>8}")
print("-" * 30)
print(f"{'Clean audio':<20} {wer_clean:>7.2f}%")
print(f"{'Noisy audio (10dB)':<20} {wer_noisy:>7.2f}%")
print(f"{'WER increase':<20} {wer_noisy - wer_clean:>7.2f}%")


Robustness Experiment (N=20, SNR=10dB noise)
Condition                 WER
------------------------------
Clean audio            37.14%
Noisy audio (10dB)     72.38%
WER increase           35.24%


In [33]:
# ============================================================
# Cell 12: Gradio Demo — Arabic ASR Studio
# ============================================================
import gradio as gr
import librosa, numpy as np, re

# Emotion emoji map (self-contained — does not depend on ED-5)
_EMOTION_EMOJI = {
    "happy":   "😊",
    "sad":     "😢",
    "angry":   "😠",
    "fearful": "😨",
    "neutral": "😐",
}

# --- Text normalization (reuse from Cell 5) ---
def normalize_arabic_text(text):
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'\u0640', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return ' '.join(text.split()).strip()

# --- Keyword detection ---
ARABIC_WORD_RE = re.compile(r'[\u0600-\u06FF]+')

DEFAULT_KEYWORDS = [
    "طوارئ",   # emergency
    "نجدة",    # help / sos
    "خطر",     # danger
    "حريق",    # fire
    "حادث",    # accident
    "امتحان",  # exam
    "اختبار",  # test
    "موعد",    # appointment
    "تسليم",   # submission
    "درس",     # lesson
    "محاضرة",  # lecture
    "واجب",    # homework
    "مشروع",   # project
    "اجتماع",  # meeting
    "مهم",     # important
    "عاجل",    # urgent
    "مستشفى",  # hospital
    "طبيب",    # doctor
    "دواء",    # medicine
    "بحث",     # research
    "تقرير",   # report
    "مقابلة",  # interview
]

def _light_stem(token: str) -> str:
    prefixes = ("وال", "بال", "كال", "فال", "لل", "ال", "و", "ف", "ب", "ك", "ل")
    suffixes = ("كما", "هما", "هم", "هن", "ها", "نا", "كم", "كن", "ات", "ان", "ين", "ون", "ه", "ة", "ي", "ك", "ا")
    for p in prefixes:
        if token.startswith(p) and len(token) - len(p) >= 2:
            token = token[len(p):]
            break
    for s in suffixes:
        if token.endswith(s) and len(token) - len(s) >= 2:
            token = token[:-len(s)]
            break
    return token

def highlight_keywords(text, keywords):
    """Return list of found keywords."""
    norm_text = normalize_arabic_text(text)
    tokens = ARABIC_WORD_RE.findall(norm_text)
    token_set = set(tokens)
    stem_set = {_light_stem(t) for t in tokens}
    found = []
    seen = set()
    for kw in keywords:
        kw_norm = normalize_arabic_text(kw)
        if not kw_norm:
            continue
        kw_stem = _light_stem(kw_norm)
        if kw_norm in token_set or kw_stem in stem_set:
            if kw not in seen:
                found.append(kw)
                seen.add(kw)
    return found

# --- Simple extractive summary ---
def simple_summary(text, n=2):
    """Return first n sentences of text as a summary."""
    sents = re.split(r'[.،؟!]', text)
    sents = [s.strip() for s in sents if len(s.strip()) > 5]
    return ' ... '.join(sents[:n]) if sents else text[:200]

# --- Main function called by Gradio ---
def run_asr(audio_path, extra_keywords, do_summary):
    if audio_path is None:
        return "Please upload or record an audio file.", "", ""

    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = (audio / np.abs(audio).max()).astype(np.float32)

    # Transcribe
    result = model.transcribe(
        audio,
        language="ar",
        task="transcribe",
        temperature=0.0,
        fp16=(device == "cuda"),
    )
    transcript = result["text"].strip()

    # ── Emotion detection ──────────────────────────────────────────────
    try:
        emo_label, emo_proba = predict_emotion(audio)
        emo_emoji = _EMOTION_EMOJI.get(emo_label, "")
        top3 = sorted(emo_proba.items(), key=lambda x: -x[1])[:3]
        top3_str = " | ".join(f"{_EMOTION_EMOJI.get(e,"")} {e}: {p*100:.1f}%" for e, p in top3)
        emotion_output = f"{emo_emoji} {emo_label}\n\n{top3_str}"
    except Exception as _emo_err:
        emotion_output = f"Emotion detection failed: {_emo_err}"

    # Keyword spotting
    all_kws = DEFAULT_KEYWORDS.copy()
    if extra_keywords:
        all_kws += [k.strip() for k in extra_keywords.split(',') if k.strip()]
    found = highlight_keywords(transcript, all_kws)
    kw_output = ", ".join(found) if found else "No keywords detected"

    # Optional summary
    summary = simple_summary(transcript) if do_summary else ""

    return transcript, kw_output, summary, emotion_output

# --- Build the interface ---
custom_css = """
.gradio-container {
  background: #F6F7FB;
}
#title {
  text-align: center;
  font-weight: 700;
}
#subtitle {
  text-align: center;
  color: #4B5563;
  margin-top: -0.5rem;
}
"""

with gr.Blocks(
    title="Arabic Speech Recognition Studio",
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate", neutral_hue="slate"),
    css=custom_css,
) as demo:

    gr.Markdown("# Arabic Speech Recognition Studio", elem_id="title")
    gr.Markdown(
        "Upload or record audio to generate a transcript, detect keywords, and optionally produce a short summary.",
        elem_id="subtitle",
    )

    with gr.Row():
        audio_input = gr.Audio(
            sources=["upload", "microphone"],
            type="filepath",
            label="Audio input (upload or record)"
        )

    with gr.Row():
        extra_kw = gr.Textbox(
            label="Additional keywords (comma-separated)",
            placeholder="Example: emergency, report, interview"
        )
        do_summary = gr.Checkbox(label="Generate summary", value=False)

    run_btn = gr.Button("Transcribe audio", variant="primary")

    with gr.Row():
        transcript_out = gr.Textbox(
            label="Transcript",
            lines=6
        )

    with gr.Row():
        kw_out      = gr.Textbox(label="Detected keywords")
        summary_out = gr.Textbox(label="Summary", lines=3)

    with gr.Row():
        emotion_out = gr.Textbox(
            label="Detected Emotion 🎭",
            lines=3,
            placeholder="Emotion will appear here after transcription..."
        )

    run_btn.click(
        fn=run_asr,
        inputs=[audio_input, extra_kw, do_summary],
        outputs=[transcript_out, kw_out, summary_out, emotion_out]
    )

    gr.Markdown("---")
    gr.Markdown("**Models:** OpenAI Whisper | **Dataset:** Mozilla Common Voice Arabic")

# share=True gives you a public URL valid for 72 hours
demo.launch(share=True, debug=False)


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://44cd3ac65c5765d183.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
